# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.

**Rule**

population --> keyword article with search volume >= 20 and low clicks/search volume ratio. a page is worth flagging if the page has recoverable opportunity

**signal**
1. search volume
- median = 10,
- mean = 146.08,
- max = 246,000,
- p10/p25 = 0.

by content type:
- keyword article --> (111,197 rows): real variation, 37% zero, usable signal
- comparison article (1,661 rows): 99.9% confirmed literal zero (not missing), 0% missing on the field itself
- feedly article (1,589 rows): 100% missing (NaN) for search_volume AND competition_level; only 2.7% missing word_count

Verdict = MIXED

2. staleness (days_since_updated_capped)
- 82.6% missing overall,
- Bucket check (mean): fresh_0_90d = −0.657 (n=19,756), aging_91_365d = −1.552 (n=208), missing = −2.408 (n=94,483)
- Bucket check (median): fresh = 0.270, aging = 0.105, missing = 0.136
- Median comparison (fresh vs. aging) points opposite of the hypothesis, but aging's n=208 is too small to trust
- Missing-staleness rows are 97.4% keyword article
- Verdict: MIXED (small-n caveat on aging bucket)


**ratio floor decision**
- Nonzero search_volume ratio check by bucket showed max ratio 95.6 (5–10 range) and 54.55 (10–20 range) which means implausible, capturing more clicks than total demand
- Floor chosen: search_volume >= 20


**Reason codes (5 total)**

- low_capture_rate — keyword article, search_volume ≥ 20 (scored)
- zero_search_volume — keyword article, search_volume = 0
- low_volume_unreliable — keyword article, 0 < search_volume < 20
- no_search_demand — comparison article
- no_demand_data_available — feedly article

**Score formula (scored rows only):**

capture_ratio = clicks / search_volume, lower ratio = higher priority

**Extra context column (not a reason code, not used for scoring):**

- staleness_context: "recently updated" / "updated over a year ago" / "no update date on record"
- Caveat to state explicitly: shown for reviewer context only, since the staleness signal check was MIXED

**Action label**

review_for_recoverable_opportunity

In [1]:
%pip -q install duckdb
import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_token")
os.environ["HF_token"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [2]:
df = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    ),
    tiered AS (
        SELECT *,
            CASE
                WHEN weighted_avg_position <= 3 THEN 'tier_1_1-3'
                WHEN weighted_avg_position <= 10 THEN 'tier_2_4-10'
                WHEN weighted_avg_position <= 20 THEN 'tier_3_11-20'
                ELSE 'tier_4_21plus'
            END AS position_tier
        FROM rollup
    ),
    benchmark(position_tier, expected_ctr) AS (
        VALUES
            ('tier_1_1-3', 0.002811), ('tier_2_4-10', 0.002353),
            ('tier_3_11-20', 0.002226), ('tier_4_21plus', 0.000748)
    ),
    labeled AS (
        SELECT
            t.content_hash_id, t.impressions, t.clicks, t.weighted_avg_position,
            b.expected_ctr,
            (b.expected_ctr * t.impressions) - t.clicks AS lost_clicks
        FROM tiered t
        JOIN benchmark b USING (position_tier)
    )
    SELECT
        l.content_hash_id, l.lost_clicks, l.impressions, l.clicks,
        l.weighted_avg_position, l.expected_ctr,
        c.search_volume, c.competition_level, c.content_type,
        CASE WHEN c.content_updated_date <= DATE '2026-03-31'
             THEN DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')
             ELSE NULL END AS days_since_update_capped,
        c.word_count
    FROM labeled l
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON l.content_hash_id = c.content_hash_id
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
df.columns.tolist()

['content_hash_id',
 'lost_clicks',
 'impressions',
 'clicks',
 'weighted_avg_position',
 'expected_ctr',
 'search_volume',
 'competition_level',
 'content_type',
 'days_since_update_capped',
 'word_count']

In [4]:
df['search_volume'].describe()

,search_volume
count,111905.0
mean,146.078281
std,2047.170943
min,0.0
25%,0.0
50%,10.0
75%,30.0
max,246000.0


In [5]:
df['search_volume'].quantile([0.1,0.25,0.5,0.75,0.9])

,search_volume
0.10,0
0.25,0
0.50,10
0.75,30
0.90,110


In [6]:
df['search_volume'].isna().sum()

np.int64(2542)

In [7]:
df.groupby('content_type')['search_volume'].apply(lambda s: (s==0).mean())

,search_volume
content_type,
comparison article,0.999398
feedly article,<NA>
keyword article,0.37251


In [8]:
df['content_type'].value_counts(dropna=False)

,count
content_type,
keyword article,111197
comparison article,1661
feedly article,1589


In [9]:
df[df['content_type'] == 'feedly article']['search_volume'].isna().sum()

np.int64(1589)

In [10]:
df[df['content_type'] == 'feedly article']['search_volume'].value_counts(dropna=False)

,count
search_volume,
<NA>,1589


In [11]:
df[df['content_type'] == 'feedly article'][['word_count', 'competition_level', 'days_since_update_capped']].isna().mean()

,0
word_count,0.027061
competition_level,1.000000
days_since_update_capped,0.589050


In [12]:
df[df['content_type'] == 'comparison article']['search_volume'].isna().sum()

np.int64(0)

In [13]:
df[df['content_type'] == 'comparison article']['search_volume'].value_counts(dropna=False)

,count
search_volume,
0,1660
10,1


In [14]:
df[df['content_type'] == 'comparison article'][['word_count', 'competition_level', 'days_since_update_capped']].isna().mean()

,0
word_count,0.000000
competition_level,0.000000
days_since_update_capped,0.910295


In [19]:
import pandas as pd
df['staleness_bucket'] = pd.cut(
    df['days_since_update_capped'],
    bins=[-1, 90, 365, float('inf')],
    labels=['fresh_0_90d', 'aging_91_365d', 'stale_365plus']
)
df['staleness_bucket'] = df['staleness_bucket'].astype('object')
df.loc[df['days_since_update_capped'].isna(), 'staleness_bucket'] = 'missing'

df.groupby('staleness_bucket')['lost_clicks'].agg(['mean', 'count'])

,mean,count
staleness_bucket,,
aging_91_365d,-1.551996,208
fresh_0_90d,-0.656947,19756
missing,-2.407830,94483


In [20]:
df[df['days_since_update_capped'].isna()]['content_type'].value_counts(normalize=True)

,proportion
content_type,
keyword article,0.974091
comparison article,0.016003
feedly article,0.009907


In [21]:
df.groupby('staleness_bucket')['lost_clicks'].agg(['median', 'mean', 'count'])

,median,mean,count
staleness_bucket,,,
aging_91_365d,0.105468,-1.551996,208
fresh_0_90d,0.270028,-0.656947,19756
missing,0.136136,-2.407830,94483


In [22]:
kw = df[df['content_type'] == 'keyword article']

kw['search_volume'].describe()

,search_volume
count,110244.0
mean,148.27909
std,2062.45625
min,0.0
25%,0.0
50%,10.0
75%,30.0
max,246000.0


In [23]:
kw['search_volume'].quantile([0.10, 0.25, 0.40, 0.50, 0.75, 0.90])

,search_volume
0.10,0
0.25,0
0.40,10
0.50,10
0.75,30
0.90,110


In [24]:
(kw['search_volume'] == 0).sum(), len(kw)

(np.int64(41067), 111197)

In [25]:
kw_pos = kw[kw['search_volume'] > 0].copy()
kw_pos['capture_ratio'] = kw_pos['clicks'] / kw_pos['search_volume']

kw_pos.groupby(pd.cut(kw_pos['search_volume'], bins=[0, 5, 10, 20, 50, 100, float('inf')]))['capture_ratio'].agg(['median', 'mean', 'max', 'count'])

/tmp/ipykernel_1540/659892267.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  kw_pos.groupby(pd.cut(kw_pos['search_volume'], bins=[0, 5, 10, 20, 50, 100, float('inf')]))['capture_ratio'].agg(['median', 'mean', 'max', 'count'])


,median,mean,max,count
search_volume,,,,
"(0.0, 5.0]",<NA>,<NA>,<NA>,0
"(5.0, 10.0]",0.1,0.665259,95.6,30451
"(10.0, 20.0]",0.05,0.410159,54.55,10031
"(20.0, 50.0]",0.025,0.213827,16.3,11975
"(50.0, 100.0]",0.011111,0.106191,27.844444,4651
"(100.0, inf]",0.0,0.027604,14.533333,12069


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [26]:
#import library
import numpy as np
import pandas as pd


In [28]:
#implementing reason code
def assign_reason_code(row):
  if row['content_type']=='comparison article':
    return 'no_search_demand'
  elif row['content_type']=='feedly article':
    return 'no_demand_data_available'
  elif row['content_type']=='keyword article':
    if pd.isna(row['search_volume']) or row['search_volume']== 0:
      return 'zero_search_volume'
    elif row['search_volume'] < 20:
      return 'low_volume_unreliable'
    else:
      return 'low_capture_rate'
  return 'unclassified_content_type'

df['reason_code']=df.apply(assign_reason_code,axis=1)

In [29]:
#scoring only for scored population
df['capture_ratio']=np.where(df['reason_code'] =='low_capture_rate', df['clicks']/df['search_volume'],np.nan)

In [31]:
#staleness context
def staleness_context(days):
  if pd.isna(days):
    return 'no update date on record'
  elif days <=365:
    return 'recently updated'
  else:
    return 'updated over a year ago'

df['staleness_context']=df['days_since_update_capped'].apply(staleness_context)

In [32]:
#action label only for scored rows
df['action_label']=np.where(df['reason_code']=='low_capture_rate','review_for_recoverable_opportunity',None)

In [33]:
#rank where lower capture ratio = more opportunity = higher priority = higher rank
df['rank']=df['capture_ratio'].rank(method='min',ascending=True)

In [35]:
#sanity check
assert df['reason_code'].eq('unclassified_content_type').sum()==0, "some rows got no reason code"

In [36]:
#final output
output_cols =['content_hash_id','content_type','search_volume','clicks','capture_ratio','reason_code','staleness_context','action_label','rank']
output = df[output_cols].sort_values('rank',na_position='first')


In [38]:
import os
os.makedirs('work/outputs',exist_ok=True)
output.to_csv('work/outputs/baseline_action_score.csv',index=False)

In [39]:
output.head(10)

,content_hash_id,content_type,search_volume,clicks,capture_ratio,reason_code,staleness_context,action_label,rank
2,content_36c36abc7650d7af,keyword article,10,6.0,NaN,low_volume_unreliable,no update date on record,None,NaN
3,content_a7da352b73b02668,keyword article,10,13.0,NaN,low_volume_unreliable,no update date on record,None,NaN
9,content_f71459b346aba398,keyword article,<NA>,1.0,NaN,zero_search_volume,no update date on record,None,NaN
12,content_30fc0ffeed8d67e6,keyword article,0,18.0,NaN,zero_search_volume,no update date on record,None,NaN
18,content_4cd532e5e9edc02b,keyword article,10,10.0,NaN,low_volume_unreliable,no update date on record,None,NaN
20,content_a76e55b3f7a9f192,keyword article,10,0.0,NaN,low_volume_unreliable,no update date on record,None,NaN
23,content_41d6b1eaf68d018d,keyword article,10,0.0,NaN,low_volume_unreliable,no update date on record,None,NaN
24,content_cb0d77f160e8b890,keyword article,10,0.0,NaN,low_volume_unreliable,no update date on record,None,NaN
25,content_80e5105d7079364d,keyword article,10,1.0,NaN,low_volume_unreliable,no update date on record,None,NaN
31,content_ede6352b9b0296e4,keyword article,10,0.0,NaN,low_volume_unreliable,no update date on record,None,NaN


In [40]:
df['reason_code'].value_counts(dropna=False)

,count
reason_code,
zero_search_volume,42020
low_capture_rate,38726
low_volume_unreliable,30451
no_search_demand,1661
no_demand_data_available,1589


In [41]:
df['capture_ratio'].notna().sum()

np.int64(38726)

In [42]:
df['rank'].isna().sum()

np.int64(75721)

In [43]:
df['rank'].dtype

dtype('float64')

In [44]:
output['rank'].head(10)

,rank
2,NaN
3,NaN
9,NaN
12,NaN
18,NaN
20,NaN
23,NaN
24,NaN
25,NaN
31,NaN


In [45]:
output = df[output_cols].sort_values('rank', na_position='last').reset_index(drop=True)
output.head(10)

,content_hash_id,content_type,search_volume,clicks,capture_ratio,reason_code,staleness_context,action_label,rank
0,content_7e9b1ea0b59a2023,keyword article,20,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
1,content_b4225b7aa5c7b32f,keyword article,70,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
2,content_ea07ff2b9bc27ea9,keyword article,110,0.0,0.0,low_capture_rate,recently updated,review_for_recoverable_opportunity,1.0
3,content_a78c34e18112e0ef,keyword article,90,0.0,0.0,low_capture_rate,recently updated,review_for_recoverable_opportunity,1.0
4,content_170a5607a9abbe19,keyword article,40,0.0,0.0,low_capture_rate,recently updated,review_for_recoverable_opportunity,1.0
5,content_895d8be04fec2e6f,keyword article,30,0.0,0.0,low_capture_rate,recently updated,review_for_recoverable_opportunity,1.0
6,content_45d8046a7620d9f6,keyword article,20,0.0,0.0,low_capture_rate,recently updated,review_for_recoverable_opportunity,1.0
7,content_14fb76a6faed7731,keyword article,30,0.0,0.0,low_capture_rate,recently updated,review_for_recoverable_opportunity,1.0
8,content_90caf920d130733a,keyword article,20,0.0,0.0,low_capture_rate,recently updated,review_for_recoverable_opportunity,1.0
9,content_dabb4886a47c61b4,keyword article,30,0.0,0.0,low_capture_rate,recently updated,review_for_recoverable_opportunity,1.0


In [46]:
(df['capture_ratio'] == 0).sum()

np.int64(17400)

In [47]:
output = df[output_cols].sort_values(
    ['rank', 'search_volume'],
    ascending=[True, False],
    na_position='last'
).reset_index(drop=True)

output.head(10)

,content_hash_id,content_type,search_volume,clicks,capture_ratio,reason_code,staleness_context,action_label,rank
0,content_ac0c525eb243379b,keyword article,246000,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
1,content_a31c400b511b1458,keyword article,201000,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
2,content_e88504a4c6d64b79,keyword article,201000,0.0,0.0,low_capture_rate,recently updated,review_for_recoverable_opportunity,1.0
3,content_0184167e6037fbc7,keyword article,201000,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
4,content_7e6779733b1dd409,keyword article,165000,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
5,content_ff42f4a65f10744c,keyword article,135000,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
6,content_6011e836cf18643a,keyword article,110000,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
7,content_d65ca6a7e33f6659,keyword article,110000,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
8,content_9755ef5214465568,keyword article,110000,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0
9,content_e7e56b5396c93880,keyword article,110000,0.0,0.0,low_capture_rate,no update date on record,review_for_recoverable_opportunity,1.0


In [48]:
import os
os.makedirs('work/outputs', exist_ok=True)
output.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Wrote {len(output)} rows")
output['reason_code'].value_counts()

Wrote 114447 rows


,count
reason_code,
zero_search_volume,42020
low_capture_rate,38726
low_volume_unreliable,30451
no_search_demand,1661
no_demand_data_available,1589


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [49]:
output.head(20)[['content_hash_id', 'search_volume', 'clicks', 'reason_code', 'staleness_context']]

,content_hash_id,search_volume,clicks,reason_code,staleness_context
0,content_ac0c525eb243379b,246000,0.0,low_capture_rate,no update date on record
1,content_a31c400b511b1458,201000,0.0,low_capture_rate,no update date on record
2,content_e88504a4c6d64b79,201000,0.0,low_capture_rate,recently updated
3,content_0184167e6037fbc7,201000,0.0,low_capture_rate,no update date on record
4,content_7e6779733b1dd409,165000,0.0,low_capture_rate,no update date on record
5,content_ff42f4a65f10744c,135000,0.0,low_capture_rate,no update date on record
6,content_6011e836cf18643a,110000,0.0,low_capture_rate,no update date on record
7,content_d65ca6a7e33f6659,110000,0.0,low_capture_rate,no update date on record
8,content_9755ef5214465568,110000,0.0,low_capture_rate,no update date on record
9,content_e7e56b5396c93880,110000,0.0,low_capture_rate,no update date on record


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.